<a href="https://colab.research.google.com/github/nubar-mamedova/mena-energy-market-expansion/blob/main/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# MENA Solar Investment Prioritization — Data Exploration
# Goal: Score 6 MENA countries (Morocco, Egypt, Saudi Arabia, UAE,
# Jordan, Turkey) for utility-scale solar investment priority.
#
# Scoring dimensions:
#  1. Solar resource quality (from Global Solar Atlas)
#  2. Market demand (from World Bank — GDP, population, electricity)
#  3. Policy & regulation (later, hand-coded from IRENA/IEA notes)
#  4. Existing renewable infrastructure (later, from OurWorldInData)
#
# Data sources: ESMAP/World Bank Global PV Potential by Country (2020),
# World Bank Open Data per-country indicators (2024).

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("environment ready")
print(f"pandas version: {pd.__version__}")

environment ready
pandas version: 2.2.2


In [1]:
# Mount Google Drive so we can read our data files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# Path to your data folder in Google Drive
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/mena-energy-market-expansion/data/raw"

# List all files in the data folder to confirm everything is there
files = os.listdir(DATA_PATH)
print(f"Found {len(files)} files:")
for f in sorted(files):
    size_kb = os.path.getsize(os.path.join(DATA_PATH, f)) / 1024
    print(f"  {f} ({size_kb:.0f} KB)")

Found 7 files:
  gsa_country_pvpotential_global.xlsx (126 KB)
  wb_country_indicators_egypt.xls (1720 KB)
  wb_country_indicators_jordan.xls (1638 KB)
  wb_country_indicators_morocco.xls (1714 KB)
  wb_country_indicators_saudiarabia.xls (1551 KB)
  wb_country_indicators_turkey.xls (1698 KB)
  wb_country_indicators_uae.xls (1442 KB)


In [4]:
import pandas as pd

# Read the GSA country-level PV potential dataset
gsa_path = os.path.join(DATA_PATH, "gsa_country_pvpotential_global.xlsx")
gsa = pd.read_excel(gsa_path, sheet_name="Country indicators", header=1)

# Show shape and column overview
print(f"GSA dataset shape: {gsa.shape}")
print(f"\nFirst few columns:")
print(gsa.columns[:15].tolist())
print(f"\nFirst 3 rows of key columns:")
print(gsa[['ISO_A3', 'Country or region']].head(10))

GSA dataset shape: (209, 21)

First few columns:
['ISO_A3', 'Country or region', 'Note', 'World Bank \nRegion', 'Total population, 2018', 'Total area, 2018', 'Evaluated area', 'Level 1 area \n(% of evaluated area)', 'Human development \nIndex, 2017', 'Gross domestic product (USD per capita), 2018', 'Average theoretical potential (GHI, kWh/m2/day), \nlong-term', 'Average practical potential \n(PVOUT Level 1, \nkWh/kWp/day), long-term', 'Average economic potential (LCOE, USD/kWh), 2018', 'Average PV \nseasonality index, long-term', 'PV equivalent area (% of total area), long-term']

First 3 rows of key columns:
  ISO_A3      Country or region
0    ABW          Aruba (Neth.)
1    AFG            Afghanistan
2    AGO                 Angola
3    ALB                Albania
4    AND                Andorra
5    ARE   United Arab Emirates
6    ARG              Argentina
7    ARM                Armenia
8    ASM  American Samoa (U.S.)
9    ATG    Antigua and Barbuda


In [6]:
# Find the solar-related columns
solar_cols = [col for col in gsa.columns if 'theoretical' in str(col).lower() or 'practical' in str(col).lower()]
print("Solar-related columns found:")
for col in solar_cols[:10]:
    print(f"  - {repr(col)}")

Solar-related columns found:
  - 'Average theoretical potential (GHI, kWh/m2/day), \nlong-term'
  - 'Average practical potential \n(PVOUT Level 1, \nkWh/kWp/day), long-term'


In [5]:
# Define our 6 target countries by ISO code
TARGET_COUNTRIES = {
    'MAR': 'Morocco',
    'EGY': 'Egypt',
    'SAU': 'Saudi Arabia',
    'ARE': 'United Arab Emirates',
    'JOR': 'Jordan',
    'TUR': 'Turkiye',
}

# Filter the GSA data to just our 6 countries
gsa_mena = gsa[gsa['ISO_A3'].isin(TARGET_COUNTRIES.keys())].copy()
print(f"Filtered to {len(gsa_mena)} countries:")
print(gsa_mena[['ISO_A3', 'Country or region']])

Filtered to 6 countries:
    ISO_A3       Country or region
5      ARE    United Arab Emirates
56     EGY  Arab Republic of Egypt
91     JOR                  Jordan
113    MAR                 Morocco
159    SAU            Saudi Arabia
189    TUR                  Turkey


In [9]:
# The exact solar column names (with embedded newlines)
SOLAR_THEORETICAL = 'Average theoretical potential (GHI, kWh/m2/day), \nlong-term'
SOLAR_PRACTICAL = 'Average practical potential \n(PVOUT Level 1, \nkWh/kWp/day), long-term'

# Also grab a few useful context columns from the GSA file
GDP_2018 = 'Gross domestic product (USD per capita), 2018'
POP_2018 = 'Total population, 2018'
AREA_2018 = 'Total area, 2018'
HDI_2017 = 'Human development \nIndex, 2017'

# Build a clean DataFrame with the columns we care about
solar_df = gsa_mena[[
    'ISO_A3',
    'Country or region',
    SOLAR_THEORETICAL,
    SOLAR_PRACTICAL,
    GDP_2018,
    POP_2018,
    AREA_2018,
    HDI_2017,
]].copy()

# Rename columns to clean, code-friendly names
solar_df.columns = [
    'iso3',
    'country_name_raw',
    'solar_theoretical_kwh_m2_day',
    'solar_practical_kwh_kwp_day',
    'gdp_per_capita_usd_2018',
    'population_2018',
    'area_sqkm_2018',
    'hdi_2017',
]

# Add our preferred display name from TARGET_COUNTRIES
solar_df['country'] = solar_df['iso3'].map(TARGET_COUNTRIES)

# Reorder columns: code, our name, then the data
solar_df = solar_df[[
    'iso3', 'country',
    'solar_theoretical_kwh_m2_day',
    'solar_practical_kwh_kwp_day',
    'gdp_per_capita_usd_2018',
    'population_2018',
    'area_sqkm_2018',
    'hdi_2017',
]]

# Sort by practical solar potential (best solar first)
solar_df = solar_df.sort_values('solar_practical_kwh_kwp_day', ascending=False).reset_index(drop=True)

print("Solar + GSA context data for the 6 MENA countries:\n")
print(solar_df.to_string(index=False))

Solar + GSA context data for the 6 MENA countries:

iso3              country  solar_theoretical_kwh_m2_day  solar_practical_kwh_kwp_day  gdp_per_capita_usd_2018  population_2018  area_sqkm_2018  hdi_2017
 JOR               Jordan                        6.0177                       5.3150              4247.768726          9956011         88780.0  0.735373
 EGY                Egypt                        6.2591                       5.2467              2549.139458         98423595        995450.0  0.695608
 SAU         Saudi Arabia                        6.2080                       5.1588             23219.130483         33699947       2149690.0  0.853299
 MAR              Morocco                        5.5632                       5.0065              3237.883368         36029138        446300.0  0.666513
 ARE United Arab Emirates                        6.0454                       5.0043             43004.948646          9630959         71020.0  0.862757
 TUR              Turkiye     

In [10]:
# Read one World Bank file to understand the structure
wb_egypt_path = os.path.join(DATA_PATH, "wb_country_indicators_egypt.xls")

# WB files have the actual headers on row 4 (zero-indexed = 3), so skiprows=3
wb_egypt = pd.read_excel(wb_egypt_path, sheet_name="Data", skiprows=3)

print(f"Shape: {wb_egypt.shape}")
print(f"\nColumn types (first 8 cols):")
print(wb_egypt.columns[:8].tolist())
print(f"\nLast 5 columns (most recent years):")
print(wb_egypt.columns[-5:].tolist())
print(f"\nFirst 5 indicators:")
print(wb_egypt[['Country Name', 'Indicator Name', 'Indicator Code']].head())

Shape: (1486, 70)

Column types (first 8 cols):
['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963']

Last 5 columns (most recent years):
['2021', '2022', '2023', '2024', '2025']

First 5 indicators:
       Country Name                                     Indicator Name  \
0  Egypt, Arab Rep.  Imports of goods and services (constant 2015 US$)   
1  Egypt, Arab Rep.             Gross capital formation (constant LCU)   
2  Egypt, Arab Rep.              Gross capital formation (current US$)   
3  Egypt, Arab Rep.           Gross fixed capital formation (% of GDP)   
4  Egypt, Arab Rep.        Gross fixed capital formation (current LCU)   

   Indicator Code  
0  NE.IMP.GNFS.KD  
1  NE.GDI.TOTL.KN  
2  NE.GDI.TOTL.CD  
3  NE.GDI.FTOT.ZS  
4  NE.GDI.FTOT.CN  


In [11]:
TARGET_INDICATORS = {
    'NY.GDP.PCAP.CD':    'gdp_per_capita_usd',
    'SP.POP.TOTL':       'population_total',
    'SP.URB.TOTL.IN.ZS': 'urban_population_pct',
    'EG.USE.ELEC.KH.PC': 'electricity_consumption_kwh_per_capita',
    'EG.ELC.ACCS.ZS':    'access_to_electricity_pct',
    'EG.ELC.RNEW.ZS':    'renewable_electricity_pct',
}

# Check which of these indicators are present in the Egypt file
found = wb_egypt[wb_egypt['Indicator Code'].isin(TARGET_INDICATORS.keys())]
print(f"Found {len(found)} of {len(TARGET_INDICATORS)} target indicators in Egypt file:\n")
print(found[['Indicator Code', 'Indicator Name']].to_string(index=False))

Found 6 of 6 target indicators in Egypt file:

   Indicator Code                                               Indicator Name
SP.URB.TOTL.IN.ZS                     Urban population (% of total population)
      SP.POP.TOTL                                            Population, total
   NY.GDP.PCAP.CD                                 GDP per capita (current US$)
EG.USE.ELEC.KH.PC                  Electric power consumption (kWh per capita)
   EG.ELC.RNEW.ZS Renewable electricity output (% of total electricity output)
   EG.ELC.ACCS.ZS                      Access to electricity (% of population)


In [13]:
# Map World Bank country file paths to ISO codes
WB_FILES = {
    'MAR': 'wb_country_indicators_morocco.xls',
    'EGY': 'wb_country_indicators_egypt.xls',
    'SAU': 'wb_country_indicators_saudiarabia.xls',
    'ARE': 'wb_country_indicators_uae.xls',
    'JOR': 'wb_country_indicators_jordan.xls',
    'TUR': 'wb_country_indicators_turkey.xls',
}

# Years to look at, in order of preference (most recent first that has data)
YEARS_PREFERENCE = ['2023', '2022', '2021', '2020']

def get_latest_value(row, years):
    """Find the most recent non-empty value from the year columns."""
    for year in years:
        val = row.get(year)
        if pd.notna(val):
            return val, year
    return None, None

# Build a list of records — one per country
records = []

for iso3, filename in WB_FILES.items():
    file_path = os.path.join(DATA_PATH, filename)
    wb_data = pd.read_excel(file_path, sheet_name="Data", skiprows=3)

    # Filter to our 6 target indicators
    wb_filtered = wb_data[wb_data['Indicator Code'].isin(TARGET_INDICATORS.keys())]

    # Build a dict for this country
    country_record = {'iso3': iso3, 'country': TARGET_COUNTRIES[iso3]}

    for _, row in wb_filtered.iterrows():
        indicator_code = row['Indicator Code']
        indicator_name = TARGET_INDICATORS[indicator_code]
        value, year = get_latest_value(row, YEARS_PREFERENCE)
        country_record[indicator_name] = value
        country_record[f"{indicator_name}_year"] = year

    records.append(country_record)

# Convert to DataFrame
wb_df = pd.DataFrame(records)

print(f"WB indicators extracted for {len(wb_df)} countries:\n")
print(wb_df.to_string(index=False))

WB indicators extracted for 6 countries:

iso3              country  electricity_consumption_kwh_per_capita electricity_consumption_kwh_per_capita_year  urban_population_pct urban_population_pct_year  population_total population_total_year  gdp_per_capita_usd gdp_per_capita_usd_year  renewable_electricity_pct renewable_electricity_pct_year  access_to_electricity_pct access_to_electricity_pct_year
 MAR              Morocco                              997.255420                                        2023             62.673751                      2023        37712505.0                  2023         3813.726074                    2023                  19.319165                           2021                      100.0                           2023
 EGY                Egypt                             1492.896036                                        2023             42.709812                      2023       114535772.0                  2023         3456.789685                    2023 

In [14]:
# Merge the GSA solar data with the WB demand/electricity data
# Both have 'iso3' column — use that as the join key
master_df = solar_df.merge(wb_df.drop(columns=['country']), on='iso3', how='left')

# Drop the redundant 2018 columns from GSA — we have fresher WB data
master_df = master_df.drop(columns=['gdp_per_capita_usd_2018', 'population_2018'])

# Reorder columns logically
column_order = [
    'iso3', 'country',
    # Solar (dimension 1)
    'solar_theoretical_kwh_m2_day',
    'solar_practical_kwh_kwp_day',
    # Demand (dimension 2)
    'gdp_per_capita_usd', 'gdp_per_capita_usd_year',
    'population_total', 'population_total_year',
    'urban_population_pct', 'urban_population_pct_year',
    'electricity_consumption_kwh_per_capita', 'electricity_consumption_kwh_per_capita_year',
    'access_to_electricity_pct', 'access_to_electricity_pct_year',
    # Existing renewables (early dimension 4 input)
    'renewable_electricity_pct', 'renewable_electricity_pct_year',
    # Context
    'area_sqkm_2018', 'hdi_2017',
]
master_df = master_df[column_order]

print("Master dataset — all dimensions in one table:\n")
print(master_df.to_string(index=False))
print(f"\nShape: {master_df.shape}")

Master dataset — all dimensions in one table:

iso3              country  solar_theoretical_kwh_m2_day  solar_practical_kwh_kwp_day  gdp_per_capita_usd gdp_per_capita_usd_year  population_total population_total_year  urban_population_pct urban_population_pct_year  electricity_consumption_kwh_per_capita electricity_consumption_kwh_per_capita_year  access_to_electricity_pct access_to_electricity_pct_year  renewable_electricity_pct renewable_electricity_pct_year  area_sqkm_2018  hdi_2017
 JOR               Jordan                        6.0177                       5.3150         4466.083142                    2023        11439213.0                  2023             92.682237                      2023                             1857.907533                                        2023                      100.0                           2023                  24.918199                           2021         88780.0  0.735373
 EGY                Egypt                        6.2591            

In [15]:
# Save the master dataset to processed folder
PROCESSED_PATH = "/content/drive/MyDrive/Colab Notebooks/mena-energy-market-expansion/data/processed"
os.makedirs(PROCESSED_PATH, exist_ok=True)

output_file = os.path.join(PROCESSED_PATH, "mena_solar_master.csv")
master_df.to_csv(output_file, index=False)
print(f"Saved {len(master_df)} rows to:\n  {output_file}")
print(f"\nFile size: {os.path.getsize(output_file) / 1024:.1f} KB")

Saved 6 rows to:
  /content/drive/MyDrive/Colab Notebooks/mena-energy-market-expansion/data/processed/mena_solar_master.csv

File size: 1.4 KB


In [18]:
# ============================================================
# SCORING MODEL (safer version — handles identical-value columns)
# ============================================================

score_df = master_df.copy()

def normalize_higher_better(series):
    """Normalize to 0-100, higher input = higher score. If all values equal, return 50 for everyone (neutral)."""
    rng = series.max() - series.min()
    if rng == 0:
        return pd.Series([50.0] * len(series), index=series.index)
    return 100 * (series - series.min()) / rng

def normalize_lower_better(series):
    """Normalize to 0-100, lower input = higher score. If all values equal, return 50 for everyone."""
    rng = series.max() - series.min()
    if rng == 0:
        return pd.Series([50.0] * len(series), index=series.index)
    return 100 * (series.max() - series) / rng

# --- Dimension 1: Solar resource ---
score_df['score_solar'] = normalize_higher_better(score_df['solar_practical_kwh_kwp_day'])

# --- Dimension 2: Market demand ---
score_df['gdp_total_usd'] = score_df['gdp_per_capita_usd'] * score_df['population_total']
score_df['_demand_gdp_total']   = normalize_higher_better(score_df['gdp_total_usd'])
score_df['_demand_electricity'] = normalize_higher_better(score_df['electricity_consumption_kwh_per_capita'])
score_df['_demand_population']  = normalize_higher_better(score_df['population_total'])
score_df['score_demand'] = (
    score_df['_demand_gdp_total']
    + score_df['_demand_electricity']
    + score_df['_demand_population']
) / 3

# --- Dimension 3: Renewables headroom ---
score_df['score_headroom'] = normalize_lower_better(score_df['renewable_electricity_pct'])

# --- Dimension 4: Market readiness ---
score_df['_readiness_urban']  = normalize_higher_better(score_df['urban_population_pct'])
score_df['_readiness_access'] = normalize_higher_better(score_df['access_to_electricity_pct'])
score_df['score_readiness'] = (
    score_df['_readiness_urban']
    + score_df['_readiness_access']
) / 2

# --- Final weighted score ---
WEIGHTS = {
    'solar':     0.25,
    'demand':    0.35,
    'headroom':  0.25,
    'readiness': 0.15,
}

score_df['final_score'] = (
    WEIGHTS['solar']     * score_df['score_solar']
    + WEIGHTS['demand']    * score_df['score_demand']
    + WEIGHTS['headroom']  * score_df['score_headroom']
    + WEIGHTS['readiness'] * score_df['score_readiness']
)

# Sanity check: are there any NaN values in scores?
nan_check = score_df[['score_solar','score_demand','score_headroom','score_readiness','final_score']].isna().sum()
print("NaN counts per score column (should all be 0):")
print(nan_check)
print()

# Rank
score_df['rank'] = score_df['final_score'].rank(ascending=False, method='min').astype(int)

# Show the final ranking
ranking_view = score_df[[
    'rank', 'country',
    'score_solar', 'score_demand', 'score_headroom', 'score_readiness',
    'final_score',
]].sort_values('rank')

print("MENA Solar Investment Priority Ranking\n" + "="*50)
print(ranking_view.round(1).to_string(index=False))
print(f"\nWeights applied: {WEIGHTS}")

NaN counts per score column (should all be 0):
score_solar        0
score_demand       0
score_headroom     0
score_readiness    0
final_score        0
dtype: int64

MENA Solar Investment Priority Ranking
 rank              country  score_solar  score_demand  score_headroom  score_readiness  final_score
    1         Saudi Arabia         84.3          66.2           100.0             66.6         79.2
    2 United Arab Emirates         68.7          46.8            88.3             68.0         65.8
    3                Egypt         93.1          44.3            65.7             25.0         59.0
    4               Jordan        100.0           2.3            29.7             75.0         44.5
    5              Morocco         68.9          11.4            45.5             45.0         39.3
    6              Turkiye          0.0          61.1             0.0             71.3         32.1

Weights applied: {'solar': 0.25, 'demand': 0.35, 'headroom': 0.25, 'readiness': 0.15}


In [19]:
# Save the final scored dataset
scored_file = os.path.join(PROCESSED_PATH, "mena_solar_scored.csv")
score_df.to_csv(scored_file, index=False)
print(f"Saved {len(score_df)} rows of scored data to:\n  {scored_file}")

# Also save the clean ranking view (this is what Tableau will use)
ranking_file = os.path.join(PROCESSED_PATH, "mena_solar_ranking.csv")
ranking_view.to_csv(ranking_file, index=False)
print(f"Saved ranking view to:\n  {ranking_file}")

Saved 6 rows of scored data to:
  /content/drive/MyDrive/Colab Notebooks/mena-energy-market-expansion/data/processed/mena_solar_scored.csv
Saved ranking view to:
  /content/drive/MyDrive/Colab Notebooks/mena-energy-market-expansion/data/processed/mena_solar_ranking.csv


In [20]:
# ============================================================
# SENSITIVITY ANALYSIS
# ============================================================
# Test if the ranking is robust to different weighting choices.
# We define 5 alternative weight scenarios reflecting different
# investor priorities, then re-run the scoring.
# ============================================================

SCENARIOS = {
    'baseline':      {'solar': 0.25, 'demand': 0.35, 'headroom': 0.25, 'readiness': 0.15},
    'solar_focused': {'solar': 0.50, 'demand': 0.20, 'headroom': 0.20, 'readiness': 0.10},
    'demand_focused':{'solar': 0.15, 'demand': 0.55, 'headroom': 0.15, 'readiness': 0.15},
    'opportunity':   {'solar': 0.20, 'demand': 0.20, 'headroom': 0.50, 'readiness': 0.10},
    'equal_weight':  {'solar': 0.25, 'demand': 0.25, 'headroom': 0.25, 'readiness': 0.25},
}

def compute_score(df, weights):
    """Compute final weighted score using given weights."""
    return (
        weights['solar']     * df['score_solar']
        + weights['demand']    * df['score_demand']
        + weights['headroom']  * df['score_headroom']
        + weights['readiness'] * df['score_readiness']
    )

# Build a results table: rows = countries, columns = scenarios (final score)
sensitivity_scores = pd.DataFrame({'country': score_df['country']})
sensitivity_ranks = pd.DataFrame({'country': score_df['country']})

for scenario_name, weights in SCENARIOS.items():
    final = compute_score(score_df, weights)
    sensitivity_scores[scenario_name] = final.round(1)
    sensitivity_ranks[scenario_name] = final.rank(ascending=False, method='min').astype(int)

print("Final SCORES under each weighting scenario:\n")
print(sensitivity_scores.to_string(index=False))

print("\n\nRANKINGS under each weighting scenario:\n")
print(sensitivity_ranks.to_string(index=False))

Final SCORES under each weighting scenario:

             country  baseline  solar_focused  demand_focused  opportunity  equal_weight
              Jordan      44.5           63.9            32.0         42.8          51.8
               Egypt      59.0           71.1            52.0         62.8          57.0
        Saudi Arabia      79.2           82.0            74.1         86.8          79.3
             Morocco      39.3           50.3            30.2         43.3          42.7
United Arab Emirates      65.8           68.2            59.5         74.0          67.9
             Turkiye      32.1           19.4            44.3         19.4          33.1


RANKINGS under each weighting scenario:

             country  baseline  solar_focused  demand_focused  opportunity  equal_weight
              Jordan         4              4               5            5             4
               Egypt         3              2               3            3             3
        Saudi Arabia  

In [21]:
# Which country is #1 in each scenario?
print("Top-ranked country by scenario:")
for scenario in SCENARIOS.keys():
    top = sensitivity_ranks.loc[sensitivity_ranks[scenario] == 1, 'country'].values
    print(f"  {scenario:18s} → #1: {', '.join(top)}")

# How often does Saudi Arabia hold #1?
sau_top = (sensitivity_ranks[list(SCENARIOS.keys())].loc[sensitivity_ranks['country'] == 'Saudi Arabia'].values == 1).sum()
print(f"\nSaudi Arabia held #1 rank in {sau_top}/{len(SCENARIOS)} scenarios")

# Compute average rank stability — how much each country's rank moves across scenarios
import numpy as np
sensitivity_ranks['rank_range'] = (
    sensitivity_ranks[list(SCENARIOS.keys())].max(axis=1)
    - sensitivity_ranks[list(SCENARIOS.keys())].min(axis=1)
)
print(f"\nRank stability (lower = more stable):")
print(sensitivity_ranks[['country', 'rank_range']].sort_values('rank_range').to_string(index=False))

Top-ranked country by scenario:
  baseline           → #1: Saudi Arabia
  solar_focused      → #1: Saudi Arabia
  demand_focused     → #1: Saudi Arabia
  opportunity        → #1: Saudi Arabia
  equal_weight       → #1: Saudi Arabia

Saudi Arabia held #1 rank in 5/5 scenarios

Rank stability (lower = more stable):
             country  rank_range
        Saudi Arabia           0
              Jordan           1
               Egypt           1
United Arab Emirates           1
             Morocco           2
             Turkiye           2


In [22]:
# Save sensitivity results
sensitivity_scores.to_csv(os.path.join(PROCESSED_PATH, "mena_solar_sensitivity_scores.csv"), index=False)
sensitivity_ranks.to_csv(os.path.join(PROCESSED_PATH, "mena_solar_sensitivity_ranks.csv"), index=False)
print("Sensitivity outputs saved.")

Sensitivity outputs saved.


In [23]:
# ============================================================
# KEY FINDINGS SUMMARY
# ============================================================
print("=" * 60)
print("PROJECT FINDINGS — MENA SOLAR INVESTMENT PRIORITIZATION")
print("=" * 60)
print()
print("1. PRIMARY RECOMMENDATION")
print("   Saudi Arabia is the #1 investment priority across all")
print("   5 weighting scenarios tested. Driven by:")
print("   - High solar resource (84/100)")
print("   - Large market demand (66/100)")
print("   - Near-zero current renewable share (0.07% → 100/100 headroom)")
print()
print("2. SECONDARY PRIORITIES")
print("   UAE (consistent #2-3) and Egypt (consistent #2-3) form")
print("   the strong secondary tier. Choice depends on whether")
print("   investor prioritizes execution efficiency (UAE) or")
print("   market scale (Egypt).")
print()
print("3. TURKEY PARADOX")
print("   Largest market by GDP+population but lowest solar resource")
print("   AND highest current renewable penetration. Only ranks well")
print("   in demand-focused scenarios. Unique profile.")
print()
print("4. ROBUSTNESS")
print("   Top-3 ranking is stable across all tested weighting choices.")
print("   The conclusion does not depend on subjective weight selection.")

PROJECT FINDINGS — MENA SOLAR INVESTMENT PRIORITIZATION

1. PRIMARY RECOMMENDATION
   Saudi Arabia is the #1 investment priority across all
   5 weighting scenarios tested. Driven by:
   - High solar resource (84/100)
   - Large market demand (66/100)
   - Near-zero current renewable share (0.07% → 100/100 headroom)

2. SECONDARY PRIORITIES
   UAE (consistent #2-3) and Egypt (consistent #2-3) form
   the strong secondary tier. Choice depends on whether
   investor prioritizes execution efficiency (UAE) or
   market scale (Egypt).

3. TURKEY PARADOX
   Largest market by GDP+population but lowest solar resource
   AND highest current renewable penetration. Only ranks well
   in demand-focused scenarios. Unique profile.

4. ROBUSTNESS
   Top-3 ranking is stable across all tested weighting choices.
   The conclusion does not depend on subjective weight selection.
